In [1]:
from nltk.tokenize import word_tokenize
from torch.nn import Embedding
from torch import nn
import torch
import math
input = "I love cats"
token = word_tokenize(input)


In [2]:
vocabulary = {}
vocabulary.update({ 0 :"/s"})
for i in range(0, len(token)):
    vocabulary.update({i+1 : token[i]})
vocabulary.update({4 : "EOS"})
vocab = vocabulary.__len__()  


In [3]:
# embedded = nn.Embedding(token.__len__() , embedding_dim=512)
# embedded
# embedded.weight

In [4]:
class Embedd(nn.Module):

    def __init__(self , d_model  , vocab):
        super().__init__()
        self.embedded = nn.Embedding(vocab , d_model)
        self.d_model = d_model
    def forward (self, x):
        return self.embedded(x) *  math.sqrt(self.d_model)   
    

In [5]:
class PositionalEncoding(nn.Module):
    
    def __init__(self , dropout , d_model , seq_len = 500):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(seq_len , d_model)
        position = torch.arange(0. , seq_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0 , d_model ,2) * - (math.log(10000.0)/d_model) )
        pe[: , 0::2] = torch.sin(position * div_term)
        pe[: , 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)

        self.register_buffer('pe' , pe)
    def forward(self , x):
        x = x + (self.pe[: , :x.shape[1] , : ]).requires_grad_(False)
        return self.dropout(x)




In [6]:
class LayerNormalization(nn.Module) :
    def __init__(self , eps=1e-6):
        super().__init__()
        self.gamma = nn.parameter(torch.ones(1))
        self.beta = nn.parameter(torch.zeros(1))
        self.eps = eps
    def forward(self , x):
        mean = x.mean(-1 , keepdim=True)
        std = x.std(-1 , keepdim=True)
        return self.gammas * (x - mean) / (std + self.eps) + self.beta     

In [ ]:
class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model , d_ff , dropout=0.1):
        super().__init__()
        self.w1 = nn.Linear(d_model , d_ff) # bias automatically is true
        self.w2 = nn.Linear(d_ff , d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self , x):
        return self.w2(self.droput(torch.ReLU(self.w1(x))))        

In [ ]:
class Attentions(nn.Module):

    def __init__(self , Q , K , V , d_model , sequence_len):
        super().__init__()
        self.Q = Q
        self.K = K
        self.V = V
        attention = torch.zeros(sequence_len , d_model)
        dot_product = Q @ torch.transpose(K)
        first_part = dot_product / torch.pow(d_model , 0.5)
        attention = torch.softmax(first_part)
        attention = attention @ V
        def forward(self , x ) :
            return attention(x) + x 